# Code Agent (Notebook)

A generic agentic assistant that can interact with GitHub repos, Jira issues, and more via MCP tools. Pass any query and the agent will break it down into steps, call the necessary tools, and complete the task autonomously.

- An **OpenAI-compliant** inference endpoint secured with API Key
- **MCP Endpoint** secured with API Key — tools for GitHub (PRs, branches, file ops) and Jira (issues, search)

Configure via `.env`: `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `MCP_SERVER_URL`, `MCP_AUTH_TOKEN`, optionally `OPENAI_MODEL`.

## 1. Imports and config

In [ ]:
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MCP_URL = os.getenv("MCP_SERVER_URL")
MCP_BEARER_TOKEN = os.getenv("MCP_AUTH_TOKEN")
OPENAI_BASE = os.getenv("OPENAI_BASE_URL")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-oss-20b")

print(f"MCP_URL: {MCP_URL}")
print(f"OPENAI_BASE: {OPENAI_BASE or '(not set)'}")
print(f"OPENAI_MODEL: {OPENAI_MODEL}")

## 2. MCP client

In [ ]:
# Session ID from server; capture from initialize (and any) response, send on all subsequent requests.
MCP_SESSION_ID = [None]


def _mcp_headers():
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if MCP_BEARER_TOKEN:
        headers["Authorization"] = f"Bearer {MCP_BEARER_TOKEN}"
    if MCP_SESSION_ID[0]:
        headers["Mcp-Session-Id"] = MCP_SESSION_ID[0]
    return headers


def _capture_session_id(resp) -> None:
    """Extract MCP session ID from response headers or body and store for subsequent requests."""
    sid = resp.headers.get("Mcp-Session-Id") or resp.headers.get("mcp-session-id")
    if not sid and (resp.text or "").strip():
        try:
            data = _parse_mcp_response_body(resp.text)
            result = data.get("result") if isinstance(data, dict) else data
            if isinstance(result, dict):
                sid = result.get("sessionId") or result.get("session_id")
        except Exception:
            pass
    if sid:
        MCP_SESSION_ID[0] = sid


def _parse_mcp_response_body(text: str) -> dict:
    """Parse MCP response: plain JSON or SSE (event: message, data: {...})."""
    if not text or not text.strip():
        return {}
    text = text.strip()
    # Plain JSON
    if text.startswith("{"):
        return json.loads(text)
    # SSE: lines like "event: message" and "data: {...}"
    if "data:" in text:
        data_parts = []
        for line in text.split("\n"):
            line = line.strip()
            if line.startswith("data:"):
                data_parts.append(line[5:].strip())
        if data_parts:
            json_str = "\n".join(data_parts)
            if json_str:
                return json.loads(json_str)
    return json.loads(text)


def mcp_request(method: str, params: dict | None = None) -> dict:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method}
    if params is not None:
        payload["params"] = params
    headers = _mcp_headers()
    resp = requests.post(MCP_URL, json=payload, headers=headers, timeout=30)
    _capture_session_id(resp)
    resp.raise_for_status()
    try:
        data = _parse_mcp_response_body(resp.text or "")
    except json.JSONDecodeError:
        raise RuntimeError(f"MCP response is not JSON. Status: {resp.status_code}. Body: {(resp.text or '')[:300]!r}")
    if "error" in data:
        raise RuntimeError(data["error"].get("message", data["error"]))
    return data.get("result", {})


def mcp_initialize():
    return mcp_request("initialize", {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "code-review-agent", "version": "1.0"},
    })


def mcp_list_tools() -> list:
    result = mcp_request("tools/list")
    all_tools = result.get("tools", [])
    return all_tools

def mcp_call_tool(name: str, arguments: dict) -> str:
    result = mcp_request("tools/call", {"name": name, "arguments": arguments})
    for part in result.get("content", []):
        if part.get("type") == "text":
            return part.get("text", "")
    return ""

## 3. MCP → OpenAI tools and system prompt

In [ ]:
def mcp_tools_to_openai(mcp_tools: list) -> list:
    openai_tools = []
    for t in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": t.get("inputSchema", {"type": "object", "properties": {}}),
            },
        })
    return openai_tools


SYSTEM_PROMPT = """You are a helpful AI agent with access to tools for interacting with GitHub repositories, Jira issues, and more.

When the user asks you to accomplish a task, break it down into steps and use the available tools to complete each step. You may need to call multiple tools in sequence — do so until the task is fully complete. After finishing, reply with a concise summary of what you did."""

## 4. Generic agent loop

In [ ]:
def _agent_loop(client, messages, openai_tools, max_turns=15):
    """Inner loop: let the LLM call tools across multiple turns until done."""
    final_content = ""
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto",
        )
        choice = response.choices[0]
        msg = choice.message

        print(f"\n--- Turn {turn + 1} (finish_reason={choice.finish_reason}) ---")
        if msg.content:
            print(f"Assistant: {msg.content}")
            final_content = msg.content.strip()

        tool_calls = getattr(msg, "tool_calls", None) or []
        if not tool_calls:
            return final_content or "Done."

        for tc in tool_calls:
            print(f"  🔧 {tc.function.name}({tc.function.arguments[:200]})")

        assistant_msg = {
            "role": "assistant",
            "content": msg.content or None,
            "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in tool_calls
            ],
        }
        messages.append(assistant_msg)

        for tc in tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            except json.JSONDecodeError:
                args = {}
            try:
                result = mcp_call_tool(name, args)
            except Exception as e:
                result = json.dumps({"error": str(e)})
            print(f"  ✅ {name} → {result[:300]}...")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return final_content or "Max turns reached."


def interactive_chat(max_turns_per_query: int = 15):
    """
    Interactive chat loop: continuously prompts for user input,
    runs the agent loop for each query, and keeps conversation history
    so follow-up questions work. Type 'exit' or 'quit' to stop.
    """
    if not OPENAI_BASE:
        raise ValueError("Set OPENAI_API_BASE to your OpenAI-compliant inference endpoint URL")

    base_url = OPENAI_BASE.rsplit("/chat/completions", 1)[0] if "/chat/completions" in OPENAI_BASE else OPENAI_BASE.rstrip("/")
    client = OpenAI(base_url=base_url, api_key=OPENAI_KEY)

    mcp_initialize()
    mcp_tool_list = mcp_list_tools()
    openai_tools = mcp_tools_to_openai(mcp_tool_list)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    print("💬 Interactive Agent Chat (type 'exit' to quit)")
    print("=" * 50)

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            break

        messages.append({"role": "user", "content": user_input})
        result = _agent_loop(client, messages, openai_tools, max_turns=max_turns_per_query)
        messages.append({"role": "assistant", "content": result})

## 5. Run the agent

In [ ]:
interactive_chat()